# S-FFSD Ensemble Model Correlation

This notebook measures how similar the three selected ensemble candidates are on the same `S-FFSD` test rows:

- `LightGBM`
- `AdaBoost`
- `Event-Based GNN`

The main objective is to check whether the models are complementary enough to justify ensembling. High correlation means the models behave similarly; lower correlation means they may add different signal.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MODEL_ROOT = PROJECT_ROOT / 'models'

PREDICTION_FILES = {
    'lightgbm': MODEL_ROOT / 'ssfd_lightgbm' / 'ssfd_lightgbm_test_predictions.csv',
    'adaboost': MODEL_ROOT / 'ssfd_adaboost' / 'ssfd_adaboost_test_predictions.csv',
    'event_gnn': MODEL_ROOT / 'ssfd_event_gnn' / 'ssfd_event_gnn_test_predictions.csv',
}

for model_name, file_path in PREDICTION_FILES.items():
    print(f'{model_name}: {file_path} | exists={file_path.exists()}')

In [ ]:
KEY_COLUMNS = ['Time', 'Source', 'Target', 'Amount', 'Location', 'Type', 'Labels']

prediction_frames = {}
for model_name, file_path in PREDICTION_FILES.items():
    df = pd.read_csv(file_path)
    df = df[KEY_COLUMNS + ['predicted_probability', 'predicted_label']].copy()
    df = df.rename(
        columns={
            'predicted_probability': f'{model_name}_probability',
            'predicted_label': f'{model_name}_label',
        }
    )
    prediction_frames[model_name] = df

base = prediction_frames['lightgbm'].copy()
for model_name in ['adaboost', 'event_gnn']:
    base = base.merge(prediction_frames[model_name], on=KEY_COLUMNS, how='inner')

expected_rows = len(prediction_frames['lightgbm'])
print(f'Merged rows: {len(base):,}')
print(f'Expected rows from LightGBM file: {expected_rows:,}')
print(f'All rows aligned: {len(base) == expected_rows}')
base.head()

In [ ]:
score_columns = [
    'lightgbm_probability',
    'adaboost_probability',
    'event_gnn_probability',
]

pearson_corr = base[score_columns].corr(method='pearson')
spearman_corr = base[score_columns].corr(method='spearman')

print('Pearson correlation')
display(pearson_corr)

print('Spearman correlation')
display(spearman_corr)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, matrix, title in zip(
    axes,
    [pearson_corr, spearman_corr],
    ['Pearson score correlation', 'Spearman score correlation'],
):
    image = ax.imshow(matrix.values, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_title(title)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f'{matrix.iloc[i, j]:.3f}', ha='center', va='center', color='black')

fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.85)
plt.tight_layout()
plt.show()

In [ ]:
label_columns = [
    'lightgbm_label',
    'adaboost_label',
    'event_gnn_label',
]

label_agreement = pd.DataFrame(index=label_columns, columns=label_columns, dtype=float)
for left in label_columns:
    for right in label_columns:
        label_agreement.loc[left, right] = (base[left] == base[right]).mean()

print('Predicted-label agreement rate')
display(label_agreement)

In [ ]:
fraud_only = base[base['Labels'] == 1].copy()
normal_only = base[base['Labels'] == 0].copy()

fraud_corr = fraud_only[score_columns].corr(method='pearson')
normal_corr = normal_only[score_columns].corr(method='pearson')

print(f'Fraud rows: {len(fraud_only):,}')
display(fraud_corr)

print(f'Normal rows: {len(normal_only):,}')
display(normal_corr)

In [ ]:
summary = pd.DataFrame(
    {
        'model': ['lightgbm', 'adaboost', 'event_gnn'],
        'mean_score': [base['lightgbm_probability'].mean(), base['adaboost_probability'].mean(), base['event_gnn_probability'].mean()],
        'std_score': [base['lightgbm_probability'].std(), base['adaboost_probability'].std(), base['event_gnn_probability'].std()],
        'positive_rate': [base['lightgbm_label'].mean(), base['adaboost_label'].mean(), base['event_gnn_label'].mean()],
    }
)
summary

## Interpretation guide

- If all three probability correlations are extremely high, the ensemble will likely add limited diversity.
- If one model is consistently less correlated with the other two, it is the main source of ensemble diversity.
- Label agreement is useful operationally: two models can have correlated scores but still disagree after thresholding.
- The fraud-only correlation block is especially important because ensemble quality depends more on how differently the models rank true fraud cases than on how similarly they score easy normal rows.